# JagNet Runner

This notebook serves as the runner for the DeepONet used for the JAG ICF dataset for ECE 228 ML for Physical Applications final project.

by Rochan Yakkundi, Ryle Rel, Matthew Idso, Bella Gennuso

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from DeepONet import train_DeepONet, model_evaluate, invariance_predict, make_coordinates, fourier_encode
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device("cuda:0" if torch.cuda.is_available else "cpu")
if torch.cuda.is_available():
    print(f"Default device set to: {torch.cuda.get_device_name(torch.cuda.current_device())}")

# Data Loader

In [ ]:
scalar_path = Path('icf-jag-10k/jag10K_0_scalars.npy')
image_path = Path('icf-jag-10k/jag10K_images.npy')
params_path = Path('icf-jag-10k/jag10K_params.npy')

scalars = np.load(scalar_path)
images = np.load(image_path)
params = np.load(params_path)

print(scalars.shape)
print(images.shape)
print(params.shape)



params_torch, scalars_torch, images_torch = torch.tensor(params, dtype=torch.float32), torch.tensor(scalars, dtype=torch.float32), torch.tensor(images, dtype=torch.float32) 

N = scalars.shape[0]
P = 50  # number of basis dimensions
h =64; w=64; c=4  #(height, width, num_channels)


dataset = TensorDataset(params_torch, images_torch, scalars_torch)
train_size = int(0.8 * N)
test_size = params.shape[0] - train_size
train_params, test_params = random_split(dataset, [train_size, test_size])  # Train test split
test_params_torch = torch.tensor(params[test_params.indices], dtype=torch.float32)
test_images_torch = torch.tensor(images[test_params.indices], dtype=torch.float32)
test_scalars_torch = torch.tensor(scalars[test_params.indices], dtype=torch.float32)

print(test_params_torch.shape)

# Training the Model

In [ ]:
num_epochs = 1000
trained_model, train_loss, test_loss = train_DeepONet(params_torch, images_torch, scalars_torch, num_epochs, 
                               branch_depth=3, trunk_depth=5, scalar_depth=3, hidden_size=1000, 
                               learn_rate=1e-4, output_freq=10, p=P, h=h, w=w, c=c, lambda_s=1.0)

# Evaluating the Model

In [ ]:
# Call model evaluate function. You can put one or multiple samples for params_eval, images_eval, and scalars_eval
print(test_params_torch.shape)
sample_index = 202  # can make any sample index in the dataset

params_eval = test_params_torch[sample_index,:].unsqueeze(0)
images_eval = test_images_torch[sample_index,:].unsqueeze(0)
scalars_eval = test_scalars_torch[sample_index,:].unsqueeze(0)

random_params = torch.tensor([0.5, 0.5, 0.5, 0.5, 0.5], dtype=torch.float32).unsqueeze(0)  # true invariance test input

img_pred, sca_pred, scalar_r2, imag_r2 = model_evaluate(trained_model, params_eval, images_eval, scalars_eval, h, w, c)  # for ground model evaluation test (complete our project)
img_invar, sca_invar = invariance_predict(trained_model, random_params, h=20, w=40, c=4)  # for true invariance test (additional feature to be worked)
# Loss Function Curve

epoch_list = np.arange(num_epochs)

fig, ax = plt.subplots(figsize=(8,6))

train_loss_np = np.array(train_loss); test_loss_np = np.array(test_loss)
ax.semilogy(epoch_list, train_loss_np, label='Training Loss')
ax.semilogy(epoch_list, test_loss_np, label='Test Image Loss')
ax.grid(); ax.legend()
ax.set_title('Train and Test Loss Curves'); ax.set_xlabel('Number of Epochs'); ax.set_ylabel('Log (Loss Value)')

print(img_invar.shape, sca_invar.shape)

# Visualizing Benchmark Test

In [ ]:
test_images_reshaped = images_eval.reshape((images_eval.shape[0], h, w, c))

print(sca_pred.shape)

if sca_pred.shape[0] > 1:
    sample_index = sample_index
else:
    sample_index = 0
    
fig, axes = plt.subplots(2, c, figsize=(4*c, 8))
for ch in range(c):
    axes[0, ch].imshow(test_images_reshaped[sample_index, :, :, ch], cmap='viridis')
    axes[0, ch].set_title(f'True Channel # {ch}')
    axes[1, ch].imshow(img_pred[sample_index, :, :, ch], cmap='viridis')
    axes[1, ch].set_title(f'Predicted Channel {ch}')
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    ax.scatter(scalars_eval[:, i], sca_pred[:, i], alpha=0.3, s=5)
    mn = min(scalars_eval[:, i].min(), sca_pred[:, i].min())
    mx = max(scalars_eval[:, i].max(), sca_pred[:, i].max())
    ax.plot([mn, mx], [mn, mx], 'r--')   # perfect prediction line
    ax.set_title(f'Scalar {i}')
    ax.set_xlabel('True'); ax.set_ylabel('Pred')
    ax.grid()
plt.tight_layout()
plt.show()

# Visualizing Invariance Test

In [ ]:

print(sca_pred.shape)
    
fig, axes = plt.subplots(2, c, figsize=(4*c, 8))
for ch in range(c):
    axes[0, ch].imshow(test_images_reshaped[sample_index, :, :, ch], cmap='viridis')
    axes[0, ch].set_title(f'True Channel # {ch}')
    axes[1, ch].imshow(img_invar[sample_index, :, :, ch], cmap='viridis')
    axes[1, ch].set_title(f'Predicted Channel {ch}')
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    ax.scatter(scalars_eval[:, i], sca_invar[:, i], alpha=0.3, s=5)
    mn = min(scalars_eval[:, i].min(), sca_invar[:, i].min())
    mx = max(scalars_eval[:, i].max(), sca_invar[:, i].max())
    ax.plot([mn, mx], [mn, mx], 'r--')   # perfect prediction line
    ax.set_title(f'Scalar {i}')
    ax.set_xlabel('True'); ax.set_ylabel('Pred')
    ax.grid()
plt.tight_layout()
plt.show()